## 1、数据处理

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from collections import Counter

### 1.1 导入数据

In [2]:
# 加载数据
data_path = '../data/news.csv'  # 更新为正确的路径
data = pd.read_csv(data_path)

In [3]:
data

,label,text
0,教育,澳移民子女成长记：带着中国心融入主流社会新华网悉尼5月31日电 无论哪个国家的父辈与子女间都...
1,体育,快船vs火箭首发：休城旨在练兵 小德帕特森进先发新浪体育讯北京时间4月10日消息，在常规赛还...
2,科技,3英寸屏高清闪存DV 三洋TH1特价1499 作者：中关村在线 飘雪 ...
3,时尚,贝嫂乏味归乏味 还有人买账Victoria Beckham 乏味归乏味 还有人买账大姐大的阵...
4,房产,三亚岭南赶房集 金九银十再兴购房游纯粹的旅行闲适有余却“收获”不足，设计一条可以兼容曼妙风景...
...,...,...
995,娱乐,组图：本-斯蒂勒与布莱克助阵《寻找伴郎》首映新浪娱乐讯 北京时间3月18日(美国当地时间3月...
996,房产,"房源阶段性不足价格高涨 楼市将上演新一轮疯狂近日，中海地产(企业专区,旗下楼盘)以70.06..."
997,科技,宽屏广角高清DC！佳能110IS仅售1980【山东IT在线报道】佳能IXUS 110 IS装...
998,时政,公安部建成打拐DNA数据库通缉50名人贩●不到1个月，侦破拐卖儿童、妇女案件逾300起，解救...


In [4]:
data['label']

0      教育
1      体育
2      科技
3      时尚
4      房产
       ..
995    娱乐
996    房产
997    科技
998    时政
999    体育
Name: label, Length: 1000, dtype: str

In [5]:
data['label'].unique()

<ArrowStringArray>
['教育', '体育', '科技', '时尚', '房产', '家居', '财经', '时政', '娱乐', '游戏']
Length: 10, dtype: str

### 1.2 创建词汇表

In [6]:
# 1. 数据准备
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.freq_threshold = freq_threshold

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in sentence:
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        return [self.stoi[token] if token in self.stoi else self.stoi["<UNK>"] for token in text]

class NewsDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts.iloc[index]
        label = self.labels.iloc[index]
        numericalized_text = [self.vocab.stoi["<SOS>"]] + self.vocab.numericalize(text)[:self.max_length-2] + [self.vocab.stoi["<EOS>"]]
        padded_text = numericalized_text + [self.vocab.stoi["<PAD>"]] * (self.max_length - len(numericalized_text))
        return torch.tensor(padded_text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [7]:
tokens = data['text'].apply(list)
tokens

0      [澳, 移, 民, 子, 女, 成, 长, 记, ：, 带, 着, 中, 国, 心, 融, ...
1      [快, 船, v, s, 火, 箭, 首, 发, ：, 休, 城, 旨, 在, 练, 兵, ...
2      [3, 英, 寸, 屏, 高, 清, 闪, 存, D, V,  , 三, 洋, T, H, ...
3      [贝, 嫂, 乏, 味, 归, 乏, 味,  , 还, 有, 人, 买, 账, V, i, ...
4      [三, 亚, 岭, 南, 赶, 房, 集,  , 金, 九, 银, 十, 再, 兴, 购, ...
                             ...                        
995    [组, 图, ：, 本, -, 斯, 蒂, 勒, 与, 布, 莱, 克, 助, 阵, 《, ...
996    [房, 源, 阶, 段, 性, 不, 足, 价, 格, 高, 涨,  , 楼, 市, 将, ...
997    [宽, 屏, 广, 角, 高, 清, D, C, ！, 佳, 能, 1, 1, 0, I, ...
998    [公, 安, 部, 建, 成, 打, 拐, D, N, A, 数, 据, 库, 通, 缉, ...
999    [全, 场, 打, 铁, 4, 4, 次, 也, 能, 赢, 球, ？,  , 公, 牛, ...
Name: text, Length: 1000, dtype: object

In [ ]:
vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(data['text'].apply(list)) # pandas的apply函数会将每个文本转换为一个单词列表，然后传递给build_vocabulary方法来构建词汇表。

In [9]:
vocab.itos

{0: '<PAD>',
 1: '<SOS>',
 2: '<EOS>',
 3: '<UNK>',
 4: '，',
 5: '中',
 6: '国',
 7: '的',
 8: '澳',
 9: '大',
 10: '对',
 11: '不',
 12: '。',
 13: '民',
 14: '子',
 15: '珍',
 16: '妮',
 17: '父',
 18: '学',
 19: '女',
 20: '语',
 21: '和',
 22: '文',
 23: '华',
 24: '社',
 25: '会',
 26: '、',
 27: '化',
 28: '是',
 29: '在',
 30: '融',
 31: '入',
 32: '主',
 33: '流',
 34: '一',
 35: '他',
 36: '同',
 37: '人',
 38: '新',
 39: '利',
 40: '亚',
 41: '们',
 42: '母',
 43: '家',
 44: '教',
 45: '育',
 46: '要',
 47: '生',
 48: '说',
 49: '代',
 50: '以',
 51: '系',
 52: '关',
 53: '时',
 54: '快',
 55: '有',
 56: '发',
 57: '：',
 58: '体',
 59: ' ',
 60: '1',
 61: '价',
 62: '9',
 63: '0',
 64: '所',
 65: '下',
 66: '这',
 67: '了',
 68: '常',
 69: '机',
 70: '为',
 71: '3',
 72: '5',
 73: '重',
 74: '三',
 75: '种',
 76: '者',
 77: '2',
 78: 'D',
 79: '还',
 80: '用',
 81: '英',
 82: '寸',
 83: '等',
 84: '4',
 85: '最',
 86: 'm',
 87: '感',
 88: '光',
 89: '像',
 90: '能',
 91: '值',
 92: '洋',
 93: 'T',
 94: 'H',
 95: '可',
 96: '7',
 97: '.',
 98: '摄',
 99:

In [10]:
vocab.stoi

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 '<UNK>': 3,
 '，': 4,
 '中': 5,
 '国': 6,
 '的': 7,
 '澳': 8,
 '大': 9,
 '对': 10,
 '不': 11,
 '。': 12,
 '民': 13,
 '子': 14,
 '珍': 15,
 '妮': 16,
 '父': 17,
 '学': 18,
 '女': 19,
 '语': 20,
 '和': 21,
 '文': 22,
 '华': 23,
 '社': 24,
 '会': 25,
 '、': 26,
 '化': 27,
 '是': 28,
 '在': 29,
 '融': 30,
 '入': 31,
 '主': 32,
 '流': 33,
 '一': 34,
 '他': 35,
 '同': 36,
 '人': 37,
 '新': 38,
 '利': 39,
 '亚': 40,
 '们': 41,
 '母': 42,
 '家': 43,
 '教': 44,
 '育': 45,
 '要': 46,
 '生': 47,
 '说': 48,
 '代': 49,
 '以': 50,
 '系': 51,
 '关': 52,
 '时': 53,
 '快': 54,
 '有': 55,
 '发': 56,
 '：': 57,
 '体': 58,
 ' ': 59,
 '1': 60,
 '价': 61,
 '9': 62,
 '0': 63,
 '所': 64,
 '下': 65,
 '这': 66,
 '了': 67,
 '常': 68,
 '机': 69,
 '为': 70,
 '3': 71,
 '5': 72,
 '重': 73,
 '三': 74,
 '种': 75,
 '者': 76,
 '2': 77,
 'D': 78,
 '还': 79,
 '用': 80,
 '英': 81,
 '寸': 82,
 '等': 83,
 '4': 84,
 '最': 85,
 'm': 86,
 '感': 87,
 '光': 88,
 '像': 89,
 '能': 90,
 '值': 91,
 '洋': 92,
 'T': 93,
 'H': 94,
 '可': 95,
 '7': 96,
 '.': 97,
 '摄': 98,
 '出'

### 1.3 数据划分

In [ ]:
label_to_int = {label: index for index, label in enumerate(data['label'].unique())}
data['label'] = data['label'].map(label_to_int) # pandas的map函数会根据提供的映射关系将原始标签转换为整数索引。
train_texts, test_texts, train_labels, test_labels = train_test_split(data['text'], data['label'], test_size=0.2) # sklearn的train_test_split函数会将数据集划分为训练集和测试集，其中test_size参数指定了测试集占总数据的比例。

In [12]:
label_to_int

{'教育': 0,
 '体育': 1,
 '科技': 2,
 '时尚': 3,
 '房产': 4,
 '家居': 5,
 '财经': 6,
 '时政': 7,
 '娱乐': 8,
 '游戏': 9}

In [13]:
data['label']

0      0
1      1
2      2
3      3
4      4
      ..
995    8
996    4
997    2
998    7
999    1
Name: label, Length: 1000, dtype: int64

### 1.4 创建DataLoader

In [ ]:
max_length = 256
train_dataset = NewsDataset(train_texts, train_labels, vocab, max_length) # 创建训练数据集，NewsDataset类会将文本转换为数值索引，并进行适当的填充以确保每个输入序列的长度一致。
test_dataset = NewsDataset(test_texts, test_labels, vocab, max_length) # 创建测试数据集，NewsDataset类会将文本转换为数值索引，并进行适当的填充以确保每个输入序列的长度一致。
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True) # PyTorch的DataLoader类会根据提供的数据集和批量大小创建一个可迭代的数据加载器，其中shuffle参数指定是否在每个epoch开始时打乱数据。
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False) # 创建测试数据加载器，通常不需要打乱测试数据，因此shuffle参数设置为False。

## 2、创建模型

### 2.1 定义模型

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # 嵌入层会将输入的单词索引转换为密集的向量表示，vocab_size是词汇表的大小，embedding_dim是每个单词嵌入向量的维度。
        print("self.embedding: ",self.embedding)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True) # batch_first=True表示输入和输出的形状为(batch_size, seq_length, feature_dim)，而不是默认的(seq_length, batch_size, feature_dim)。
        print("self.rnn: ",self.rnn)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim) # 如果是双向RNN，隐藏状态的维度会翻倍，因此需要乘以2。
        print("self.fc: ",self.fc)
        self.dropout = nn.Dropout(dropout) # dropout层会在训练过程中随机将一部分神经元的输出设置为0，以防止过拟合。 
        print("self.dropout: ",self.dropout)

    def forward(self, text):
        #print("embedded: ",self.embedding(text).shape)
        embedded = self.dropout(self.embedding(text)) # 首先将输入文本转换为嵌入向量，然后应用dropout进行正则化。
        print("embedded: ",embedded.shape)
        output, hidden = self.rnn(embedded) # RNN层会处理嵌入向量序列，并返回输出和隐藏状态。output的形状为(batch_size, seq_length, hidden_dim * num_directions)，hidden的形状为(num_layers * num_directions, batch_size, hidden_dim)。
        print("output: ",output.shape)
        print("hidden: ",hidden.shape)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)) # 如果是双向RNN，我们需要将正向和反向的隐藏状态拼接在一起，然后应用dropout进行正则化。
        else:
            hidden = self.dropout(hidden[-1,:,:]) # 如果是单向RNN，我们只需要使用最后一层的隐藏状态，并应用dropout进行正则化。
            
        print("hidden.shape: ",hidden.shape)
        return self.fc(hidden) # 最后通过全连接层将隐藏状态映射到输出维度，得到最终的分类结果。

### 2.2 模型超参数

In [ ]:
# 模型参数
vocab_size = len(vocab.stoi)
embedding_dim = 100
hidden_dim = 256
output_dim = len(label_to_int)
n_layers = 2
bidirectional = False
dropout = 0.5
model = RNN(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout) # 创建RNN模型实例，传入词汇表大小、嵌入维度、隐藏层维度、输出维度、层数、是否双向和dropout率等参数。

self.embedding:  Embedding(2970, 100)
self.rnn:  RNN(100, 256, num_layers=2, batch_first=True, dropout=0.5)
self.fc:  Linear(in_features=256, out_features=10, bias=True)


In [17]:
next(iter(train_loader))

[tensor([[   1,  386,  347,  ...,  331,   38,    2],
         [   1,   60,   72,  ...,  195,   12,    2],
         [   1,  630,  833,  ...,   29, 1673,    2],
         ...,
         [   1,  109,  105,  ...,  562,   68,    2],
         [   1,  940,   81,  ...,  193,  102,    2],
         [   1,  630,  833,  ...,    7,  336,    2]]),
 tensor([6, 8, 3, 5, 3, 3, 8, 5, 9, 3, 4, 4, 6, 8, 3, 0, 6, 7, 9, 0, 7, 3, 1, 7,
         9, 5, 8, 1, 4, 1, 6, 9, 1, 3, 8, 1, 7, 2, 2, 5, 5, 5, 5, 1, 7, 6, 1, 0,
         3, 3, 3, 9, 7, 3, 1, 6, 9, 5, 1, 0, 1, 7, 7, 3])]

In [ ]:
next(iter(train_loader))[0].shape, next(iter(train_loader))[1].shape # 获取训练数据加载器中的第一个批次，返回输入文本的形状和标签的形状。输入文本的形状为(batch_size, seq_length)，标签的形状为(batch_size,)。

(torch.Size([64, 256]), torch.Size([64]))

In [19]:
model(next(iter(train_loader))[0])

embedded:  torch.Size([64, 256, 100])
output:  torch.Size([64, 256, 256])
hidden:  torch.Size([2, 64, 256])
hidden.shape:  torch.Size([64, 256])


tensor([[ 2.5422e-02, -4.5906e-01,  5.2676e-01, -2.1394e-01,  3.5266e-01,
          9.3732e-02,  9.3190e-02, -2.3375e-01,  7.0018e-01,  3.6446e-02],
        [-2.7183e-01,  4.4881e-02,  5.0133e-01,  3.6496e-01, -3.0742e-01,
         -4.8281e-01, -8.7578e-02,  1.7641e-01,  1.5460e-02,  4.4355e-02],
        [ 6.6734e-02, -9.4973e-02, -9.4317e-02,  3.4764e-01,  4.4710e-01,
          5.2778e-02,  3.3228e-01, -1.9066e-01,  3.6040e-01, -6.3687e-02],
        [ 7.5181e-01, -5.8552e-02, -1.0861e-01, -3.0183e-01,  1.4476e-02,
          3.0489e-01,  1.3122e-01, -3.6227e-01,  8.7999e-02,  3.4691e-01],
        [-4.2287e-01, -2.9134e-02,  6.2929e-02,  1.1006e-03, -2.1386e-01,
         -3.7448e-02, -8.1825e-03, -4.4957e-02,  2.3516e-01,  3.3755e-01],
        [-5.1503e-02,  9.5230e-02,  1.4708e-01,  4.4249e-01, -7.8552e-02,
         -3.1269e-02,  2.9321e-01, -1.7160e-01,  6.4242e-01,  1.8420e-01],
        [ 5.9382e-02, -3.3450e-01, -3.7433e-01,  1.7255e-01, -4.4962e-01,
          1.6584e-01,  9.4540e-0

### 2.3 配置模型

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # 嵌入层会将输入的单词索引转换为密集的向量表示，vocab_size是词汇表的大小，embedding_dim是每个单词嵌入向量的维度。
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True) # batch_first=True表示输入和输出的形状为(batch_size, seq_length, feature_dim)，而不是默认的(seq_length, batch_size, feature_dim)。
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim) # 如果是双向RNN，隐藏状态的维度会翻倍，因此需要乘以2。
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text)) # 首先将输入文本转换为嵌入向量，然后应用dropout进行正则化。
        output, hidden = self.rnn(embedded)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)) # 如果是双向RNN，我们需要将正向和反向的隐藏状态拼接在一起，然后应用dropout进行正则化。
        else:
            hidden = self.dropout(hidden[-1,:,:]) # 如果是单向RNN，我们只需要使用最后一层的隐藏状态，并应用dropout进行正则化。
        return self.fc(hidden) # 最后通过全连接层将隐藏状态映射到输出维度，得到最终的分类结果。

In [ ]:
# 模型参数
vocab_size = len(vocab.stoi)
embedding_dim = 100
hidden_dim = 256
output_dim = len(label_to_int)
n_layers = 2
bidirectional = True
dropout = 0.6
model = RNN(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout) # 创建RNN模型实例，传入词汇表大小、嵌入维度、隐藏层维度、输出维度、层数、是否双向和dropout率等参数。

In [ ]:
# 3. 训练模型
optimizer = optim.Adam(model.parameters()) # 使用Adam优化器来更新模型的参数，model.parameters()会返回模型中所有需要优化的参数。
criterion = nn.CrossEntropyLoss() # 定义损失函数，CrossEntropyLoss适用于多分类问题，它会计算模型输出的概率分布与真实标签之间的交叉熵损失。
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # 检查是否有可用的GPU，如果有则使用GPU，否则使用CPU。
model = model.to(device) # 将模型移动到指定的设备上（GPU或CPU），以便在训练过程中进行计算。
criterion = criterion.to(device) # 将损失函数也移动到同一设备上，以确保在计算损失时不会出现设备不匹配的错误。

### 2.4 定义训练函数

In [ ]:
def train(model, iterator, optimizer, criterion):
    model.train() # 将模型设置为训练模式，这会启用dropout和batch normalization等特定于训练的层。
    epoch_loss = 0
    epoch_acc = 0

    for batch in iterator:
        optimizer.zero_grad() # 在每个批次开始时清除之前的梯度，以避免梯度累积。
        predictions = model(batch[0].to(device)) # 将输入数据移动到同一设备上，并通过模型进行前向传播，得到预测结果。
        loss = criterion(predictions, batch[1].to(device)) # 计算损失，criterion会比较模型的预测结果和真实标签，并返回一个标量损失值。
        loss.backward() # 反向传播计算梯度，loss.backward()会自动计算模型中所有参数的梯度，并将其存储在每个参数的.grad属性中。
        optimizer.step() # 更新模型参数，optimizer.step()会根据计算得到的梯度来更新模型的参数。

        epoch_loss += loss.item() # 累加当前批次的损失值，以便在整个epoch结束后计算平均损失。
        # 计算准确率
        _, preds = torch.max(predictions, dim=1) # torch.max函数会返回每行的最大值和对应的索引，dim=1表示在列方向上进行操作，这里我们只需要索引来计算准确率。
        epoch_acc += torch.sum(preds == batch[1].to(device)).item() # 将预测的标签与真实标签进行比较，计算正确预测的数量，并累加到epoch_acc中。

    return epoch_loss / len(iterator), epoch_acc / len(iterator.dataset) # 返回平均损失和准确率，平均损失是总损失除以批次数，准确率是正确预测的数量除以总样本数。

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    epoch_acc = 0

    with torch.no_grad(): # 在评估模式下，我们不需要计算梯度，因此使用torch.no_grad()上下文管理器来禁用梯度计算，以节省内存和加速计算。
        for batch in iterator: # 遍历评估数据加载器中的每个批次，进行前向传播并计算损失和准确率。
            predictions = model(batch[0].to(device))
            loss = criterion(predictions, batch[1].to(device))
            epoch_loss += loss.item()
            # 计算准确率
            _, preds = torch.max(predictions, dim=1)
            epoch_acc += torch.sum(preds == batch[1].to(device)).item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator.dataset) # 返回平均损失和准确率，平均损失是总损失除以批次数，准确率是正确预测的数量除以总样本数。



### 2.5 开始训练

In [ ]:
num_epochs = 20
for epoch in range(num_epochs): # 迭代指定的epoch次数，在每个epoch中进行训练和评估。
    train_loss, train_acc = train(model, train_loader, optimizer, criterion) # 调用train函数进行训练，传入模型、训练数据加载器、优化器和损失函数，返回训练损失和准确率。
    valid_loss, valid_acc = evaluate(model, test_loader, criterion) # 调用evaluate函数进行评估，传入模型、测试数据加载器和损失函数，返回评估损失和准确率。
    
    print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val. Loss: {valid_loss:.4f}, Val. Acc: {valid_acc:.4f}')

Epoch: 1, Train Loss: 2.3440, Train Acc: 0.1300, Val. Loss: 2.1264, Val. Acc: 0.2250
Epoch: 2, Train Loss: 2.1574, Train Acc: 0.2338, Val. Loss: 1.9340, Val. Acc: 0.3450
Epoch: 3, Train Loss: 2.0060, Train Acc: 0.2825, Val. Loss: 1.8316, Val. Acc: 0.3800
Epoch: 4, Train Loss: 1.8916, Train Acc: 0.3212, Val. Loss: 1.7871, Val. Acc: 0.3900
Epoch: 5, Train Loss: 1.7866, Train Acc: 0.3837, Val. Loss: 1.8370, Val. Acc: 0.3950
Epoch: 6, Train Loss: 1.7114, Train Acc: 0.4037, Val. Loss: 1.6813, Val. Acc: 0.4200
Epoch: 7, Train Loss: 1.5945, Train Acc: 0.4600, Val. Loss: 1.4756, Val. Acc: 0.4950
Epoch: 8, Train Loss: 1.5358, Train Acc: 0.4763, Val. Loss: 1.5581, Val. Acc: 0.4700
Epoch: 9, Train Loss: 1.4185, Train Acc: 0.5375, Val. Loss: 1.7506, Val. Acc: 0.4800
Epoch: 10, Train Loss: 1.2799, Train Acc: 0.5725, Val. Loss: 1.8365, Val. Acc: 0.4650
Epoch: 11, Train Loss: 1.2417, Train Acc: 0.5700, Val. Loss: 2.1731, Val. Acc: 0.3350
Epoch: 12, Train Loss: 1.1946, Train Acc: 0.5938, Val. Loss: 1.